### 特徵工程 (Feature Engineering)

本檔案主要功能為建立特徵以供後續建模、定義 A/B 群組以供 SHAP 計算。

In [2]:
import pandas as pd
import numpy as np
import os

def create_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    """建立有意義的進階互動特徵"""
    df = df.copy()

    # 1. 季後賽表現提升幅度 (大賽屬性)
    if 'VORP_reg' in df.columns and 'VORP_playoff' in df.columns:
        df['VORP_Elevation'] = np.where(
            df['has_playoff_exp'] == 1,
            df['VORP_playoff'] - df['VORP_reg'],
            0 
        )
        
    if 'BPM_reg' in df.columns and 'BPM_playoff' in df.columns:
         df['BPM_Elevation'] = np.where(
            df['has_playoff_exp'] == 1,
            df['BPM_playoff'] - df['BPM_reg'],
            0
        )

    # 2. 季後賽上場時間佔比 (體力與教練信任度)
    if 'MP_reg' in df.columns and 'MP_playoff' in df.columns:
        total_mp_reg = df['MP_reg'] * df['G_reg']
        total_mp_playoff = df['MP_playoff'] * df['G_playoff']
        df['Playoff_MP_Ratio'] = np.where(
            total_mp_reg + total_mp_playoff > 0,
            total_mp_playoff / (total_mp_reg + total_mp_playoff),
            0
        )

    # 補上缺失值防呆
    df = df.fillna(0)
    return df

if __name__ == "__main__":
    # 1. 讀取上一階段清洗與合併好的完整資料
    input_path = '../../data/processed/merged_weighted_stats.csv'
    df = pd.read_csv(input_path)
    
    print(f"讀取原始資料成功，維度: {df.shape}")
    print("開始執行特徵工程...")
    
    # 2. 直接套用特徵工程，保持 DataFrame 完整性 (不提早切割)
    featured_df = create_interaction_features(df)
    
    # 3. 設定存檔路徑
    save_dir = r'C:\Users\user\Python\Final_test\data\processed'
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    save_path = os.path.join(save_dir, 'featured_nba_data.csv')
    
    # 4. 直接存檔！這張表現在具備了所有的 ID、Target、基礎特徵與進階特徵
    featured_df.to_csv(save_path, index=False)
    
    print(f"✅ 特徵資料庫已成功儲存至: {save_path}")
    print(f"✅ 最終資料維度: {featured_df.shape}")

讀取原始資料成功，維度: (898, 102)
開始執行特徵工程...
✅ 特徵資料庫已成功儲存至: C:\Users\user\Python\Final_test\data\processed\featured_nba_data.csv
✅ 最終資料維度: (898, 105)
